In [ ]:
import io
import random

from datetime import datetime

import pandas as pd

#2

def gerar_registro():

    return {
        "ano": 2024,
        "id_municipio": str(
            random.randint(1000000, 9999999)
        ),
        "id_escola": (
            f"E{random.randint(1000,9999)}"
        ),
        "id_aluno": (
            f"A{random.randint(100000,999999)}"
        ),
        "proficiencia": round(
            random.uniform(100, 300),
            2
        ),
        "data_ingestao": (
            datetime.utcnow().isoformat() + "Z"
        )
    }
`

# 3
def gerar_lote(qtd_registros):

    registros = []

    for _ in range(qtd_registros):

        registros.append(
            gerar_registro()
        )

    return pd.DataFrame

# 4
def upload_parquet(
    blob_service_client,
    container_name,
    dataframe,
    blob_name
):

    parquet_buffer = io.BytesIO()

    dataframe.to_parquet(
        parquet_buffer,
        index=False,
        engine="pyarrow"
    )

    blob_client = (
        blob_service_client.get_blob_client(
            container=container_name,
            blob=blob_name
        )
    )

    blob_client.upload_blob(
        parquet_buffer.getvalue(),
        overwrite=True
    )

    return len(dataframe)

#5

def execute_batch(
    blob_service_client,
    container_name,
    records,
    folder
):

    df = gerar_lote(records)

    timestamp = datetime.utcnow().strftime(
        "%Y/%m/%d/%H%M%S"
    )

    blob_name = (
        f"{folder}/"
        f"{timestamp}/"
        f"dados.parquet"
    )

    total_sent = upload_parquet(
        blob_service_client,
        container_name,
        df,
        blob_name
    )

    return {
        "records_sent": total_sent,
        "blob_name": blob_name,
        "preview": df.head()
    }